[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/experimental-psychology/blob/main/notebooks/pitch_ratings.ipynb)

# Pitch Session Ratings Analysis

This notebook provides tools for analyzing the pitch ratings data from the Pitch Session Lab. You'll compare how different groups' pitches were rated across multiple dimensions.

In [ ]:
# Cell 2: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set plot style for clean visuals
sns.set_theme(style='whitegrid')
%matplotlib inline

## Loading the Pitch Ratings Data

The class pitch ratings are stored in a [Google Sheet](https://docs.google.com/spreadsheets/d/1f4EbN5AtnR53CynJoMfJO2Ydni8-XPb8a4Y7IH0dBZw/edit?usp=sharing). The cell below loads the data directly from this sheet as CSV.

In [ ]:
# Load pitch ratings data directly from the class Google Sheet
data_url = 'https://docs.google.com/spreadsheets/d/1f4EbN5AtnR53CynJoMfJO2Ydni8-XPb8a4Y7IH0dBZw/export?format=csv'

df = pd.read_csv(data_url)

# Rename columns from Google Forms question text to short names
column_mapping = {
    'Timestamp': 'timestamp',
    "Which group's pitch are you evaluating?": 'group',
    'How CLEAR was the pitch?': 'clarity',
    'How INTERESTING was the pitch?': 'interest',
    'How EFFICIENT was the pitch?': 'efficiency',
    'How effective was the chosen FORMAT of the pitch?': 'format',
}

# Apply renaming with fuzzy matching (substring match as fallback)
renamed = {}
for old_col in df.columns:
    for pattern, new_name in column_mapping.items():
        if pattern.lower() in old_col.lower() or old_col.lower() in pattern.lower():
            renamed[old_col] = new_name
            break
    else:
        renamed[old_col] = old_col.lower().replace(' ', '_')

df = df.rename(columns=renamed)

# Drop the timestamp column (not needed for analysis)
if 'timestamp' in df.columns:
    df = df.drop(columns=['timestamp'])

# Define the rating dimensions we'll analyze
dimensions = ['clarity', 'interest', 'efficiency', 'format']

print(f'Loaded {len(df)} ratings across {df["group"].nunique()} groups')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Cell 5: Compute mean ratings per group across all dimensions
group_means = df.groupby('group')[dimensions].mean()

# Add a row for overall means across all groups
group_means.loc['Overall'] = df[dimensions].mean()

# Round for readability
summary = group_means.round(2)
print('Mean Ratings by Group')
print('=' * 50)
summary

In [ ]:
# Cell 6: Grouped bar chart — mean ratings by group for each dimension
fig, ax = plt.subplots(figsize=(10, 6))

# Remove the 'Overall' row for plotting individual groups
plot_data = group_means.drop('Overall', errors='ignore')

# Create grouped bar chart
plot_data.plot(kind='bar', ax=ax, width=0.7, edgecolor='black', linewidth=0.5)

ax.set_xlabel('Group', fontsize=12)
ax.set_ylabel('Mean Rating (1-10)', fontsize=12)
ax.set_title('Mean Pitch Ratings by Group', fontsize=14)
ax.set_ylim(0, 10.5)
ax.legend(title='Dimension', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Radar / spider chart comparing group profiles across all 4 dimensions
groups = sorted(df['group'].dropna().unique())
num_dims = len(dimensions)

# Compute angles for the radar chart (one per dimension, closing the loop)
angles = np.linspace(0, 2 * np.pi, num_dims, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

# Color palette for groups
colors = sns.color_palette('husl', len(groups))

for i, group in enumerate(groups):
    values = group_means.loc[group, dimensions].tolist()
    values += values[:1]  # close the polygon
    ax.plot(angles, values, 'o-', linewidth=2, label=f'Group {group}', color=colors[i])
    ax.fill(angles, values, alpha=0.1, color=colors[i])

# Set dimension labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels([d.capitalize() for d in dimensions], fontsize=12)
ax.set_ylim(0, 10)
ax.set_title('Group Profiles Across Rating Dimensions', fontsize=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: One-way ANOVA for each dimension
# Tests whether groups differ significantly on each rating dimension

print('One-Way ANOVA Results')
print('=' * 60)
print(f'{"Dimension":<15} {"F-statistic":>12} {"p-value":>12} {"Significant?":>14}')
print('-' * 60)

alpha = 0.05  # significance threshold

for dim in dimensions:
    # Split ratings by group for this dimension
    group_data = [group_df[dim].dropna() for _, group_df in df.groupby('group')]

    # Run one-way ANOVA
    f_stat, p_val = stats.f_oneway(*group_data)

    sig = 'Yes *' if p_val < alpha else 'No'
    print(f'{dim.capitalize():<15} {f_stat:>12.3f} {p_val:>12.4f} {sig:>14}')

print('-' * 60)
print(f'* Significant at alpha = {alpha}')

In [ ]:
# Cell 9: Composite "overall effectiveness" score and group ranking
# The composite score is the unweighted mean across all four dimensions

df['overall_effectiveness'] = df[dimensions].mean(axis=1)

# Compute mean overall effectiveness per group
ranking = (df.groupby('group')['overall_effectiveness']
           .agg(['mean', 'std', 'count'])
           .rename(columns={'mean': 'Mean', 'std': 'Std Dev', 'count': 'N Ratings'})
           .sort_values('Mean', ascending=False))

ranking['Rank'] = range(1, len(ranking) + 1)
ranking = ranking[['Rank', 'Mean', 'Std Dev', 'N Ratings']].round(2)

print('Overall Effectiveness Ranking')
print('=' * 50)
ranking

## Tips for Interpretation

As you review the results above, consider the following:

- **Clarity**: Did the audience understand the research question and approach? High clarity ratings suggest the group communicated their idea in an accessible way.
- **Interest**: Did the pitch capture attention? This reflects how compelling or novel the research idea felt to the audience.
- **Efficiency**: Was the time used well? Groups that scored high here delivered their message concisely without unnecessary filler.
- **Format**: Was the presentation itself well-structured? This covers things like slide design, speaking pace, and overall organization.

### What makes science communication effective?

Look at which groups scored highest overall and on individual dimensions. Are there trade-offs (e.g., high interest but low clarity)? What strategies did the top-ranked groups use that others could adopt?

### Going deeper with GenAI

Use a generative AI tool (e.g., ChatGPT, Claude) to explore your data further. Some ideas:
- Ask it to run post-hoc pairwise comparisons (e.g., Tukey HSD) on dimensions with significant ANOVA results.
- Have it generate a written summary of the findings for a general audience.
- Explore whether certain dimensions predict overall effectiveness more strongly than others (e.g., via regression).
- Ask it to suggest improvements to the visualizations or statistical analyses in this notebook.